In [ ]:
# larn_data_delite.ipynbを編集したプログラム．learn_data_delite.ipynbでは全ての値が0の行を削除していたが，このプログラムでは正解データが同じものを1つに統合している．
# また，learn_data_suplit.pyと同様に正解データの統合とともに，入力データも削除している．
# ※learn_data_delite.ipynbでは全てが0の値を完全に削除しているが，このプログラムでは統合なので，値が全て0のデータも1つ残っている．
# cd workspace/research2/experiment

#グローバル変数
cir = 's15850'  # 対象回路
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + 'integrated_output'  # 正解データファイル
suplit_num = None # 何個ずつ分割するか = 1つのモデルの正解データ数
suplit_num_differece_flag = False # 分割数が異なる場合にTrue
suplit_data_num = None # 分割された正解データが何個あるか = モデルの分割数
suplit_input_data_folder = cir + '分割入力データ' # 入力データを分割して保存するフォルダ
suplit_input_data_file = suplit_input_data_folder + '/' + cir + 'input'  # 分割された入力データファイル
correct_data_folder = cir + '分割正解データ2' # 正解データを分割して保存するフォルダ
correct_data_file = correct_data_folder + '/' + cir + 'integrated_output'  # 分割された正解データファイル
correct_data_delite_folder = cir + '分割正解データ削除後2'  # 正解データを削除した後のデータを保存するフォルダ
correct_data_integrated_inf_file = correct_data_delite_folder + '/' + cir + 'integrated_inf'  # 正解データの統合情報を保存するファイル
correct_data_delite_file = correct_data_delite_folder + '/' + cir + 'integrated_output'  # 正解データを統合・削除した後の正解データファイル
suplit_num_file = correct_data_folder + '/' + cir + 'suplit_num'  # 分割数を保存するファイル
delite_sum_file = correct_data_delite_folder + '/' + cir + 'delite_inf'  # 削除行数を保存するファイル
save_delite_inf_file = correct_data_delite_folder + '/' + cir + 'delite_inf_all'  # これまでの削除行数を保存するファイル
save_delite_inf_file_sub = correct_data_delite_folder + '/' + cir + 'delite_inf_all2'  # 分割された正解データごとの情報は保存しないファイル
delited_data_num_file = correct_data_delite_folder + '/' + cir + 'delited_data_num'  # 削除したデータ数を保存するファイル
delited_data_num_all_file = correct_data_delite_folder + '/' + cir + 'delited_data_num_all'  # これまでの削除したデータ数を保存するファイル

with open(suplit_num_file, 'r') as f:  # データの分割数を保存するファイルを開く
    suplit_data_num = int(f.readline().replace('\n', ''))  # データの分割数を読み込む
    print("正解データの分割数:", suplit_data_num)


# 入力データファイルを開いてデータを読み込む
with open(input_data_file) as f:
    input_data = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除


# 各分割された正解データファイルを開いてデータを読み込み、同じ正解データの行を削除し、それに伴って入力データも削除する
# その後、削除した正解データと入力データを新しいファイルに保存する
# 削除した行数の平均も計算して表示
all_delite_sum = 0
max_delite_sum = 0 #　正解データごとの削除行数の最大値
min_delite_sum = -1 #　正解データごとの削除行数の最小値(初期値が-1なのは，最初の削除行数が0のときに正しく動作しないため)
delite_sum_list = []
delited_data_num = [] # 削除後の正解データ数
for i in range(suplit_data_num):

    with open(correct_data_file + str(i)) as f:
        correct_data = [_.strip().split(",") for _ in f.readlines()]  # 各行の空白文字と改行文字を削除
        data_num = len(correct_data)
        if suplit_num is None:     # 最初のループで正解データ数が設定されていない場合、設定する
            suplit_num = data_num
        elif suplit_num != data_num:
            suplit_num_differece_flag = True

    # 行ごとに比較して同じ値なら統合・削除を行う（統合元は，最初に出現した行）
    unique_correct_data = []
    unique_input_data = []
    delite_sum = 0 #　正解データごとの削除行数
    for idx, row in enumerate(correct_data):
        if row not in unique_correct_data:
            unique_correct_data.append(row)
            unique_input_data.append(input_data[idx])
        else:
            all_delite_sum += 1
            delite_sum += 1
    
    delite_sum_list.append(delite_sum)
    delited_data_num.append(len(unique_correct_data))

    if delite_sum > max_delite_sum:
        max_delite_sum = delite_sum

    if delite_sum < min_delite_sum or min_delite_sum == -1:
        min_delite_sum = delite_sum
    # print(unique_correct_data)
    # print(unique_input_data)
    
    # 統合された正解データごとにその行番号をリストに格納（行番号は0から始まる）
    duplicate_idx = [[] for _ in range(len(unique_correct_data))]
    for idx, data in enumerate(unique_correct_data):
        for j in range(len(correct_data)):
            if data == correct_data[j]:
                duplicate_idx[idx].append(j)
    
    # print(duplicate_idx)

    
    with open(suplit_input_data_file + str(i), 'w') as f:
        for j in range(len(unique_input_data)):
            f.write(unique_input_data[j] + '\n')
    
    with open(correct_data_delite_file + str(i), 'w') as f:
        for j in range(len(unique_correct_data)):
            for k in range(len(unique_correct_data[j]) - 1):
                f.write(unique_correct_data[j][k] + ",")
            f.write(unique_correct_data[j][-1] + '\n')
    
    with open(correct_data_integrated_inf_file + str(i), 'w') as f:
        for j in range(len(duplicate_idx)):
            for k in range(len(duplicate_idx[j]) - 1):
                f.write(str(duplicate_idx[j][k]) + ",")
            f.write(str(duplicate_idx[j][-1]) + '\n')

with open(delite_sum_file, 'w') as f:
    f.write("最大削除行数:" + str(max_delite_sum) + "\n")
    f.write("最小削除行数:" + str(min_delite_sum) + "\n")
    f.write("平均削除行数:" + str(all_delite_sum / suplit_data_num) + "\n")
    f.write("各正解データごとの削除行数\n")
    for i in range(suplit_data_num):
        f.write("正解データ" + str(i) + ":" + str(delite_sum_list[i]) + "\n")

with open(save_delite_inf_file, 'a') as f:
    f.write("モデル分割数:" + str(suplit_data_num) + "\n")
    if suplit_num_differece_flag:
        f.write("1つのモデルの正解データ数:" + str(suplit_num) + "," + str(suplit_num+1) + "\n")
    else:
        f.write("1つのモデルの正解データ数:" + str(suplit_num) + "\n")
    f.write("最大削除行数:" + str(max_delite_sum) + "\n")
    f.write("最小削除行数:" + str(min_delite_sum) + "\n")
    f.write("平均削除行数:" + str(all_delite_sum / suplit_data_num) + "\n")
    f.write("各正解データごとの削除行数\n")
    for i in range(suplit_data_num):
        f.write("正解データ" + str(i) + ":" + str(delite_sum_list[i]) + "\n")
    f.write("\n\n\n")

with open(save_delite_inf_file_sub, 'a') as f:
    f.write("モデル分割数:" + str(suplit_data_num) + "\n")
    if suplit_num_differece_flag:
        f.write("1つのモデルの正解データ数:" + str(suplit_num) + "," + str(suplit_num+1) + "\n")
    else:
        f.write("1つのモデルの正解データ数:" + str(suplit_num) + "\n")
    f.write("最大削除行数:" + str(max_delite_sum) + "\n")
    f.write("最小削除行数:" + str(min_delite_sum) + "\n")
    f.write("平均削除行数:" + str(all_delite_sum / suplit_data_num) + "\n")
    f.write("\n\n")

with open(delited_data_num_file, 'w') as f:
    f.write("平均正解データ数:" + str(sum(delited_data_num) / suplit_data_num) + "\n")
    f.write("最大正解データ数:" + str(max(delited_data_num)) + "\n")
    f.write("最小正解データ数:" + str(min(delited_data_num)) + "\n")
    f.write("各正解データごとの削除後のデータ数\n")
    for i in range(suplit_data_num):
        f.write("正解データ" + str(i) + ":" + str(delited_data_num[i]) + "\n")

with open(delited_data_num_all_file, 'a') as f:
    f.write("モデル分割数:" + str(suplit_data_num) + "\n")
    f.write("平均正解データ数:" + str(sum(delited_data_num) / suplit_data_num) + "\n")
    f.write("最大正解データ数:" + str(max(delited_data_num)) + "\n")
    f.write("最小正解データ数:" + str(min(delited_data_num)) + "\n")
    f.write("各正解データごとの削除後のデータ数\n")
    for i in range(suplit_data_num):
        f.write("正解データ" + str(i) + ":" + str(delited_data_num[i]) + "\n")
    f.write("\n\n")


print("最大削除行数:", max_delite_sum)
print("最小削除行数:", min_delite_sum)
print("平均削除行数:", all_delite_sum / suplit_data_num)        
print("各正解データごとの削除行数:", delite_sum_list)
print("各正解データごとの削除行数:", delite_sum_list)
print("最大正解データ数:", max(delited_data_num))
print("最小正解データ数:", min(delited_data_num))
print("平均正解データ数:", sum(delited_data_num) / suplit_data_num)


正解データの分割数: 264
最大削除行数: 2043
最小削除行数: 2
平均削除行数: 1018.0378787878788
各正解データごとの削除行数: [1675, 1704, 14, 29, 1531, 258, 107, 15, 138, 32, 41, 104, 51, 227, 186, 74, 18, 50, 600, 386, 644, 1675, 1453, 1692, 1706, 1630, 1694, 1614, 1557, 1558, 974, 32, 1187, 916, 912, 1303, 1123, 1633, 1551, 1690, 1402, 1098, 1587, 1681, 1705, 1373, 1268, 1681, 644, 23, 495, 1015, 628, 1488, 1504, 1308, 1291, 1279, 108, 1340, 834, 1065, 432, 1452, 1461, 127, 128, 1705, 1609, 1353, 832, 1303, 434, 1700, 486, 109, 19, 311, 792, 1645, 1499, 1534, 1578, 8, 1697, 411, 1707, 1323, 1656, 658, 272, 2, 1054, 1358, 1392, 1457, 1515, 2, 4, 6, 726, 945, 286, 1503, 1619, 1171, 1665, 698, 515, 403, 599, 1348, 267, 220, 1123, 201, 1532, 493, 1716, 1788, 1916, 1367, 1894, 1522, 1268, 1258, 1144, 186, 212, 809, 708, 1483, 326, 426, 127, 71, 799, 579, 662, 497, 112, 179, 37, 323, 500, 646, 618, 711, 832, 462, 166, 499, 252, 1445, 1098, 1227, 612, 756, 444, 1387, 601, 1547, 466, 1407, 1752, 704, 957, 513, 418, 1962, 1405, 1137, 15